# Vanguard Website A/B Test Analysis

## Project Overview

Vanguard conducted an A/B test comparing its traditional online process (**Control**) with a redesigned interface (**Test**).

Both versions follow the same process:

`start → step_1 → step_2 → step_3 → confirm`

The objective of this analysis is to evaluate whether the redesigned experience performs better using the following KPIs:

- **Completion rate**
- **Time spent on each process step**
- **Error rate**

The analysis also tests whether:

1. The Test version has a significantly higher completion rate than the Control version.
2. The relative improvement in completion rate exceeds Vanguard's required **5% cost-effectiveness threshold**.

# 1. Setup and Data Loading

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest

DATA_DIR = Path("../data/raw")

## 1.1 Load the datasets

In [ ]:
# Client profiles
client_path = DATA_DIR / "df_final_demo.txt"
clients_df = pd.read_csv(client_path)

# Digital footprint data
web_part1_path = DATA_DIR / "df_final_web_data_pt_1.txt"
web_part2_path = DATA_DIR / "df_final_web_data_pt_2.txt"

web_part1_df = pd.read_csv(web_part1_path)
web_part2_df = pd.read_csv(web_part2_path)

web_df = pd.concat(
    [web_part1_df, web_part2_df],
    axis = 0,
    ignore_index = True
)

# Experiment roster
experiment_path = DATA_DIR / "df_final_experiment_clients.txt"
experiment_df = pd.read_csv(experiment_path)

In [ ]:
clients_df.head()

In [ ]:
web_df.head()

In [ ]:
experiment_df.head()

# 2. Data Cleaning

## 2.1 Client profiles

In [ ]:
# Check missing values
clients_df.isnull().sum()

In [ ]:
# Remove rows with substantial missing data
clients_df = clients_df.dropna(thresh = 7)

# Fill the remaining missing age value using the mean
clients_df['clnt_age'].fillna(
    clients_df['clnt_age'].mean(),
    inplace = True
)

clients_df.isnull().sum()

In [ ]:
# Review unique values
clients_df.nunique()

In [ ]:
# Standardise gender values
clients_df['gendr'] = clients_df['gendr'].map({
    "X":"U",
    "F":"F",
    "M":"M",
    "U":"U"
})

clients_df['gendr'].value_counts()

In [ ]:
# Convert selected columns to integer
columns = [
    "clnt_tenure_yr",
    "clnt_tenure_mnth",
    "clnt_age",
    "num_accts",
    "calls_6_mnth",
    "logons_6_mnth"
]

for column in columns:
    clients_df[column] = clients_df[column].astype(int)

clients_df.dtypes

In [ ]:
# Rename columns for readability
clients_df.columns = [
    'client_id',
    'client_tenure_yr',
    'client_tenure_month',
    'client_age',
    'gender',
    'num_accounts',
    'balance',
    'calls_6_month',
    'logons_6_month'
]

clients_df.head()

## 2.2 Digital footprint data

In [ ]:
web_df.isnull().sum()

In [ ]:
web_df['process_step'].value_counts()

In [ ]:
web_df.dtypes

## 2.3 Experiment roster

In [ ]:
experiment_df.isnull().sum()

In [ ]:
experiment_df['Variation'].value_counts()

In [ ]:
experiment_df.dtypes

# 3. Client Behavior Analysis

This section first reviews the overall client population, then focuses on clients assigned to the A/B experiment.

## 3.1 Overall client profile

In [ ]:
clients_df.describe()

The descriptive statistics above provide an overview of client age, tenure, account holdings, balances, calls, and logons.

## 3.2 Experiment population

In [ ]:
# Merge client profiles with experiment assignments
clients_experiment_df = pd.merge(
    clients_df,
    experiment_df,
    on = "client_id",
    how = "left"
)

clients_experiment_df.isna().sum()

In [ ]:
# Keep only clients assigned to Test or Control
clients_experiment_df.dropna(
    subset = ['Variation'],
    inplace = True
)

clients_experiment_df.describe()

In [ ]:
# Keep only web events belonging to experiment clients
web_experiment_df = web_df[
    web_df['client_id'].isin(
        clients_experiment_df['client_id']
    )
].copy()

web_experiment_df['date_time'] = pd.to_datetime(
    web_experiment_df['date_time']
)

print(
    "Unique experiment clients in web data:",
    web_experiment_df['client_id'].nunique()
)

print()
print(
    web_experiment_df['process_step'].value_counts()
)

## 3.3 Age groups and experiment-group characteristics

In [ ]:
# Create age groups
bins = [0,18,34,54,74,clients_df['client_age'].max()]
labels = ['0-18','19-34','35-54','55-74','>74']

clients_df['age_group'] = pd.cut(
    clients_df['client_age'],
    labels = labels,
    bins = bins,
    include_lowest = True
)

clients_experiment_df['age_group'] = pd.cut(
    clients_experiment_df['client_age'],
    labels = labels,
    bins = bins,
    include_lowest = True
)

clients_df['age_group'].value_counts()

In [ ]:
gender_age_counts = (
    clients_experiment_df
    .groupby(
        ['Variation','gender','age_group'],
        observed = False
    )['gender']
    .count()
    .sort_values(ascending = False)
)

gender_age_counts

In [ ]:
clients_experiment_df.groupby(
    ['Variation','age_group'],
    observed = False
)['age_group'].count().sort_values(ascending = False)

In [ ]:
clients_experiment_df.groupby(['Variation'])['client_age'].mean()

In [ ]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['client_tenure_yr'].mean()

In [ ]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['balance'].mean()

In [ ]:
clients_experiment_df.groupby(['Variation','age_group'], observed = False)['calls_6_month'].mean()

In [ ]:
clients_experiment_df.groupby(['Variation'])['logons_6_month'].mean()

# 4. Performance Metrics

The redesign is evaluated using three required KPIs:

1. **Time spent on each process step**
2. **Error rate**
3. **Completion rate**

## 4.1 Time spent on each process step

In [ ]:
web_experiment_df['DateTime'] = pd.to_datetime(
    web_experiment_df['date_time']
)

# Sort events by visit and timestamp
web_sorted_df = web_experiment_df.sort_values(
    by = ['visit_id', 'DateTime'],
    ascending = [True, True]
)

# Remove consecutive duplicate process steps within the same visit
web_sorted_df = web_sorted_df[
    ~(
        (web_sorted_df['visit_id'] == web_sorted_df['visit_id'].shift())
        &
        (web_sorted_df['process_step'] == web_sorted_df['process_step'].shift())
    )
]

# Calculate elapsed time between consecutive events
web_sorted_df['Time_Spent'] = (
    web_sorted_df
    .groupby('visit_id')['DateTime']
    .diff()
    .dt.total_seconds()
    .fillna(0)
)

# Calculate total recorded time for each step within each visit
visit_step_times_df = (
    web_sorted_df
    .groupby(
        ['visit_id', 'process_step']
    )['Time_Spent']
    .sum()
    .reset_index()
)

# Calculate the average recorded time for each process step
avg_step_times = (
    visit_step_times_df
    .groupby('process_step')['Time_Spent']
    .mean()
    .sort_values(ascending = False)
)

avg_step_times

In [ ]:
# Reset the index after sorting and filtering
web_sorted_df.reset_index(
    inplace = True
)

web_sorted_df.drop(
    'index',
    axis = 1,
    inplace = True
)

## 4.2 Error rate

A move from a later process step to an earlier process step is treated as an error.

In [ ]:
# Ensure events are ordered chronologically within each visit
web_sorted_df.sort_values(
    by = ['visit_id', 'DateTime'],
    inplace = True
)

# Define the expected process sequence
sequence = [
    'start',
    'step_1',
    'step_2',
    'step_3',
    'confirm'
]

# Map each process step to its position in the sequence
step_indices = {
    step: idx
    for idx, step in enumerate(sequence)
}

# Initialise the error indicator
web_sorted_df['error'] = 0

# Group events by visit
grouped = web_sorted_df.groupby(
    ['visit_id']
)

# Flag backwards movements
for _, group in grouped:
    steps = group['process_step'].tolist()

    for i in range(1, len(steps)):
        if step_indices[steps[i]] < step_indices[steps[i - 1]]:
            idx_current = group.index[i]
            web_sorted_df.loc[
                idx_current,
                'error'
            ] = 1

In [ ]:
web_sorted_df['error'].value_counts()

In [ ]:
# Original overall error-rate calculation retained
error_rate = 48842 / (235661 + 48842)

error_rate

In [ ]:
# Add experiment-group information to the cleaned web data
web_analysis_df = pd.merge(
    web_sorted_df,
    experiment_df,
    on = "client_id",
    how = "left"
)

web_analysis_df.isna().sum()

In [ ]:
# Calculate total errors for Test and Control
test_error_sum = web_analysis_df[
    web_analysis_df['Variation'] == 'Test'
]['error'].sum()

control_error_sum = web_analysis_df[
    web_analysis_df['Variation'] == 'Control'
]['error'].sum()

print(
    "Total errors for Test group:",
    test_error_sum
)

print(
    "Total errors for Control group:",
    control_error_sum
)

In [ ]:
# Calculate error rates for Test and Control
test_error_avg = (
    web_analysis_df[
        web_analysis_df['Variation'] == 'Test'
    ]['error'].sum()
    /
    web_analysis_df[
        web_analysis_df['Variation'] == 'Test'
    ]['date_time'].count()
)

control_error_avg = (
    web_analysis_df[
        web_analysis_df['Variation'] == 'Control'
    ]['error'].sum()
    /
    web_analysis_df[
        web_analysis_df['Variation'] == 'Control'
    ]['date_time'].count()
)

print(
    "Percentage of errors for Test website:",
    test_error_avg
)

print(
    "Percentage of errors for Control website:",
    control_error_avg
)

## 4.3 Completion rate

A visit is considered complete when it reaches the `confirm` step at least once. Each visit is therefore counted only once in the completion-rate numerator.

In [ ]:
# Count unique completed visits
completion_sum_test = (
    web_analysis_df[
        (web_analysis_df['Variation'] == "Test")
        &
        (web_analysis_df['process_step'] == "confirm")
    ]['visit_id']
    .nunique()
)

completion_sum_control = (
    web_analysis_df[
        (web_analysis_df['Variation'] == "Control")
        &
        (web_analysis_df['process_step'] == "confirm")
    ]['visit_id']
    .nunique()
)

print(
    f"Number of completed visits for the Test website is: {completion_sum_test}"
)

print(
    f"Number of completed visits for the Control website is: {completion_sum_control}"
)

In [ ]:
# Count total visits
num_of_visits_test = web_analysis_df[
    web_analysis_df['Variation'] == "Test"
]['visit_id'].nunique()

num_of_visits_control = web_analysis_df[
    web_analysis_df['Variation'] == "Control"
]['visit_id'].nunique()

print(
    f"Number of total visits for the Test website is: {num_of_visits_test}"
)

print(
    f"Number of total visits for the Control website is: {num_of_visits_control}"
)

In [ ]:
# Calculate visit-level completion rates
completion_rate_test = (
    completion_sum_test
    /
    num_of_visits_test
)

completion_rate_control = (
    completion_sum_control
    /
    num_of_visits_control
)

print(
    f"The completion rate of the Test website: {completion_rate_test}"
)

print(
    f"The completion rate of the Control website: {completion_rate_control}"
)

## 4.4 KPI comparison

In [ ]:
web_analysis_df.groupby(['Variation','process_step'])['Time_Spent'].mean()

# 5. Hypothesis Testing

In [ ]:
# Create a binary indicator for the confirm step
web_analysis_df['confirm'] = (
    web_analysis_df['process_step']
    .apply(
        lambda x: 1
        if x == 'confirm'
        else 0
    )
)

## 5.1 Is the Test completion rate higher?

- **H0:** `test_completion_rate <= control_completion_rate`
- **H1:** `test_completion_rate > control_completion_rate`

In [ ]:
# Calculate completion status for each Test visit
test_visits_df = (
    web_analysis_df[
        web_analysis_df["Variation"] == "Test"
    ]
    .groupby(
        "visit_id"
    )["confirm"]
    .sum()
)

# Calculate completion status for each Control visit
control_visits_df = (
    web_analysis_df[
        web_analysis_df["Variation"] == "Control"
    ]
    .groupby(
        "visit_id"
    )["confirm"]
    .sum()
)

# Convert the results to dataframes
test_visits_df = pd.DataFrame(
    test_visits_df
).reset_index()

control_visits_df = pd.DataFrame(
    control_visits_df
).reset_index()

# Convert completion counts into binary outcomes
control_visits_df['confirm'] = (
    control_visits_df['confirm']
    .apply(
        lambda x: 1
        if x >= 1
        else 0
    )
)

test_visits_df['confirm'] = (
    test_visits_df['confirm']
    .apply(
        lambda x: 1
        if x >= 1
        else 0
    )
)

test_visits_df['confirm'].value_counts()

In [ ]:
# Count completed visits
successes = [
    test_visits_df['confirm'].sum(),
    control_visits_df['confirm'].sum()
]

# Count total visits
nobs = [
    len(test_visits_df),
    len(control_visits_df)
]

# One-sided proportion z-test
z_stat, p_value = proportions_ztest(
    count = successes,
    nobs = nobs,
    alternative = 'larger'
)

print(
    f"Z-statistic: {z_stat}"
)

print(
    f"P-value: {p_value}"
)

alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis: "
        "The Test version has a significantly higher completion rate."
    )
else:
    print(
        "Fail to reject the null hypothesis: "
        "No significant improvement with the Test version."
    )

## 5.2 Does the Test version exceed the 5% cost-effectiveness threshold?

Vanguard requires the Test version to achieve at least a **5% relative increase** in completion rate.

- **H0:** `p_test / p_control <= 1.05`
- **H1:** `p_test / p_control > 1.05`

In [ ]:
# Number of completed visits
test_success = test_visits_df[
    'confirm'
].sum()

control_success = control_visits_df[
    'confirm'
].sum()

# Total number of visits
test_total = len(
    test_visits_df
)

control_total = len(
    control_visits_df
)

# Completion rates
p_test = (
    test_success
    /
    test_total
)

p_control = (
    control_success
    /
    control_total
)

# Relative completion-rate ratio
risk_ratio = (
    p_test
    /
    p_control
)

relative_uplift = (
    risk_ratio
    -
    1
)

# Standard error of the log risk ratio
se_log_rr = np.sqrt(
    (1 / test_success)
    - (1 / test_total)
    + (1 / control_success)
    - (1 / control_total)
)

# Test the 5% relative threshold
threshold = 1.05

z_stat = (
    np.log(risk_ratio)
    -
    np.log(threshold)
) / se_log_rr

p_value = st.norm.sf(
    z_stat
)

print(
    f"Test completion rate: {p_test}"
)

print(
    f"Control completion rate: {p_control}"
)

print(
    f"Risk ratio: {risk_ratio}"
)

print(
    f"Relative uplift: {relative_uplift * 100}%"
)

print(
    f"Z-statistic: {z_stat}"
)

print(
    f"P-value: {p_value}"
)

alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis: "
        "The Test version exceeds the 5% relative completion-rate threshold."
    )
else:
    print(
        "Fail to reject the null hypothesis: "
        "There is not enough evidence that the Test version exceeds "
        "the 5% relative completion-rate threshold."
    )

## 5.3 Time comparison for completed visits

The following one-sided Welch t-test compares total recorded visit time for visits containing a `confirm` step.

- **H0:** `control_time <= test_time`
- **H1:** `control_time > test_time`

In [ ]:
completed_visits_df = web_analysis_df.groupby(
    'visit_id'
).filter(
    lambda x:
    (x['process_step'] == 'confirm').any()
)

time_control = (
    completed_visits_df[
        completed_visits_df['Variation'] == "Control"
    ]
    .groupby(
        "visit_id"
    )['Time_Spent']
    .sum()
)

time_test = (
    completed_visits_df[
        completed_visits_df['Variation'] == "Test"
    ]
    .groupby(
        "visit_id"
    )['Time_Spent']
    .sum()
)

alpha = 0.05

st.ttest_ind(
    time_control,
    time_test,
    equal_var = False,
    alternative = "greater"
)

# 6. Conclusion

The final evaluation should consider all three KPIs together with the hypothesis-test results:

- Compare the Test and Control **visit-level completion rates**.
- Determine whether the Test version has a statistically higher completion rate.
- Determine whether the relative improvement exceeds Vanguard's required **5% threshold**.
- Compare the Test and Control error rates.
- Compare the recorded time results between the two versions.

The completion-rate hypothesis tests answer two different questions: whether completion improved at all, and whether the improvement was large enough to exceed the business threshold.

# 7. Optional Export Commands

The project does not save processed datasets during normal execution. The following export commands are retained only as reference.

In [ ]:
# web_analysis_df.to_csv(
#     "sorted_df_web_cleaned_exp.csv",
#     index = False
# )

# completed_visits_df.to_csv(
#     "filtered_df.csv",
#     index = False
# )

# clients_experiment_df.to_csv(
#     "df_client_exp.csv",
#     index = False
# )